# 04 — Prepare COMPASS figure data

Run this notebook after `01_preprocessing.ipynb`, `02_univariate.ipynb`, and the relevant model-result builds. This is the only Python execution stage for figures. `05_figures.Rmd` consumes its prepared files and never calls Python. No model training or figure rendering occurs here.

The default is **all ADT, pre-ADT castrate patients included**, for platinum and NEPC. Other ADT definitions remain available for the small cohort forests without generating full figure inputs for all six cohorts. Use `SCOPE = "federated"` for just the no-MSK PSA/testosterone forests; no longitudinal data or matching local results are needed in that mode.

The first run reads the selected treatment arm's longitudinal CSV and writes versioned Parquet caches and `manifest.json`. Re-running checks source identity and reuses unchanged caches. `05_figures.Rmd` checks this manifest, then prepares/caches its R survival statistics, optional GAM fits, and graphics objects. Missing or changed data require rerunning this notebook. Subsequent rendering reuses those objects.

Caches contain patient data. Keep `FIGURE_DATA_ROOT` under the protected analysis-data tree, not in the figure export directory.


In [ ]:
from pathlib import Path
import os
import sys

# Set before importing Polars. Restart the kernel if Polars was already imported
# and you need to change its thread-pool size.
CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", "4"))
os.environ.setdefault("POLARS_MAX_THREADS", str(max(1, min(4, CPUS))))

# Works from this notebook's directory or from the repository root.
HERE = Path.cwd()
SCRIPT_DIR = next((p for p in [HERE, HERE / "COMPASS" / "survival_analysis"]
                   if (p / "prepare_figure_data.py").is_file()), None)
if SCRIPT_DIR is None:
    raise FileNotFoundError("Run from the repository root or COMPASS/survival_analysis")
sys.path.insert(0, str(SCRIPT_DIR.resolve()))
import importlib
import prepare_figure_data as figure_data
figure_data = importlib.reload(figure_data)
import polars as pl

print(f"Python: {sys.executable}")
print(f"Polars: {pl.__version__}; threads: {pl.thread_pool_size()}")


## Configuration

These defaults match `05_figures.Rmd`. Both honor the `COMPASS_DATA_ROOT`, `COMPASS_FIGURE_DATA_ROOT`, `COMPASS_FIGURE_COHORTS`, `COMPASS_FIGURE_ENDPOINTS`, and `COMPASS_FIGURE_SCOPE` environment variables. If you edit the cohort/endpoint/lab choices here, select the corresponding settings in `05_figures.Rmd` as well. R can render a subset of prepared cohorts/endpoints, but cannot prepare additional inputs.

`FORCE = True` rebuilds cached tables even when source sizes/timestamps match. Use it after replacing source files while preserving file metadata. It is independent of the R image-overwrite switch.


In [ ]:
def env_flag(name, default=False):
    value = os.environ.get(name, str(default)).strip().lower()
    if value not in {"true", "false", "1", "0", "yes", "no"}:
        raise ValueError(f"{name} must be true or false")
    return value in {"true", "1", "yes"}

DATA_ROOT = Path(os.environ.get("COMPASS_DATA_ROOT", "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS"))
FIGURE_DATA_ROOT = Path(os.environ.get("COMPASS_FIGURE_DATA_ROOT", str(DATA_ROOT / "figure_data")))
ADT_COHORT_OPTIONS = [
    "adt", "adt_noprecastrate",
    "adt_metastatic_adt", "adt_metastatic_adt_noprecastrate",
    "adt_metastatic_llm", "adt_metastatic_llm_noprecastrate",
]
COHORTS = [x.strip() for x in os.environ.get("COMPASS_FIGURE_COHORTS", "adt").split(",")]
ENDPOINTS = [x.strip() for x in os.environ.get("COMPASS_FIGURE_ENDPOINTS", "nepc,platinum").split(",")]
SCOPE = os.environ.get("COMPASS_FIGURE_SCOPE", "all")
LABS = figure_data.canonical_lab_names() if env_flag("COMPASS_NON_ANDROGEN_LABS") else ["PSA", "Testosterone"]
FORCE = env_flag("COMPASS_PREPARE_OVERWRITE")

CLASSIFIER_PATH = Path("/data/gusev/USERS/jpconnor/data/LLM_annotations/LLM_NEPC_labels")
PROFILE_DATA_ROOT = Path(os.environ.get("PROFILE_DATA_PATH", "/data/gusev/USERS/jpconnor/data/PROFILE_DATA"))
METASTATIC_SOURCES = {
    "intent": str(DATA_ROOT / "mrn_lists" / "adt_intent_labels_model_cohort.csv"),
    "stage": os.environ.get("COMPASS_REGEX_STAGE_PATH", str(PROFILE_DATA_ROOT / "CANCER_ANNOTATIONS" / "CANCER_STAGE_NOTE_LEVEL.parquet")),
    "llm": os.environ.get("LLM_MET_LABELS_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/LLM_met_diagnosis/met_dx_labels.parquet"),
    "icd": str(DATA_ROOT / "prostate_icd_data.csv"),
}
FEDERATED_PATH = Path(os.environ.get("COMPASS_FEDERATED_NO_MSK_RESULTS",
    str(DATA_ROOT / "federated_results_no_MSK" / "cox_federated_univariate_adt.csv")))


In [ ]:
config = {
    "data_root": str(DATA_ROOT.resolve()),
    "cache_root": str(FIGURE_DATA_ROOT.resolve()),
    "cohorts": COHORTS,
    "endpoints": ENDPOINTS,
    "scope": SCOPE,
    "labs": LABS,
    "force": FORCE,
    "classifier_path": str(CLASSIFIER_PATH),
    "forest_cohorts": ADT_COHORT_OPTIONS,
    "forest_landmark": 180,
    "gam": env_flag("COMPASS_GAM_TRAJECTORIES"),
    "adt_intent": env_flag("COMPASS_ADT_INTENT_SUPPLEMENT"),
    "metastatic": env_flag("COMPASS_METASTATIC_LABEL_SUPPLEMENT", True),
    "metastatic_extra": os.environ.get("COMPASS_METASTATIC_EXTRA_PANELS", "").strip().lower() not in {"", "0"},
    "metastatic_sources": METASTATIC_SOURCES,
    "federated": env_flag("COMPASS_FEDERATED_NO_MSK_SUPPLEMENT", True),
    "federated_path": str(FEDERATED_PATH),
}
CONFIG_PATH = FIGURE_DATA_ROOT / "preparation_config.json"
figure_data.atomic_json(CONFIG_PATH, config)
print(f"Cohorts: {COHORTS}; endpoints: {ENDPOINTS}; scope: {SCOPE}")
print(f"Cache root: {FIGURE_DATA_ROOT}")
print(f"Saved preparation settings: {CONFIG_PATH}")


## Prepare changed data

One scan per changed treatment arm prepares:

- one row per patient with treatment/outcome metadata and all-lab record spans;
- selected canonical lab measurements;
- patient/lab/time-bin means on the original and `log1p` scales;
- pre-treatment measurement counts and binary bin availability;
- metastatic labels using the longitudinal analysis anchor, including ICD burden when enabled;
- small day-180 event-count tables across the six ADT cohorts and both endpoints, plus Stage-1 ADT-intent/LLM overlap counts for the combined cohort-comparison figures in `05`. These use the aggregated landmark CSVs and the two model-cohort label CSVs in `mrn_lists`; they do not trigger additional full cohort renders.

Cohort eligibility and the sequencing/Gleason time origins remain defined by the existing analysis inputs. The manifest fingerprints each selected cohort/endpoint result tree and the small cross-cohort/federated inputs used by R.


In [ ]:
import time
started = time.perf_counter()
manifest = figure_data.prepare(config)
print(f"Preparation/validation finished in {time.perf_counter() - started:.2f} seconds")
print(f"Manifest: {FIGURE_DATA_ROOT / 'manifest.json'}")

# Successful arms remain cached if another arm fails. Surface all reported
# errors here so a partial preparation cannot look like a complete run.
if manifest.get("errors"):
    for cell, error in manifest["errors"].items():
        print(f"FAILED {cell}: {error}")
    raise RuntimeError("Some figure inputs failed preparation; see the errors above")


## Check prepared outputs

This summary shows table sizes only, without displaying patient identifiers or measurements. Files remain under the analysis-data cache.


In [ ]:
rows = []
for arm, details in manifest["arms"].items():
    directory = Path(details["directory"])
    for table in figure_data.ARM_TABLES:
        path = directory / f"{table}.parquet"
        n_rows = pl.scan_parquet(path).select(pl.len()).collect().item()
        rows.append({"arm": arm, "table": table, "rows": n_rows,
                     "size_mb": round(path.stat().st_size / 1024**2, 2)})
if manifest.get("cohort_overview"):
    for table in ["incidence", "label_overlap"]:
        path = Path(manifest["cohort_overview"]["directory"]) / f"{table}.parquet"
        rows.append({"arm": "ADT overview", "table": table,
                     "rows": pl.scan_parquet(path).select(pl.len()).collect().item(),
                     "size_mb": round(path.stat().st_size / 1024**2, 2)})
if rows:
    display(pl.DataFrame(rows))
else:
    print("Federated-only scope: no patient tables are needed")
print("Prepared result fingerprints:", ", ".join(manifest["cells"]) or "federated comparison only")


## Next: `05_figures.Rmd`

Knit `05_figures.Rmd` with its default `COMPASS_FIGURE_STAGE=all`. R checks the manifest/settings and file metadata, prepares and caches the statistical/graphics objects, and renders missing or stale images. It never starts Python. If it reports missing, changed, or incompatible prepared inputs, rerun this notebook.

After that first R preparation:

- `COMPASS_FIGURE_STAGE=render` changes DPI/PDF formats using the cached snapshot without invoking Python or reading source patient data.
- `COMPASS_FIGURE_STAGE=prepare` refreshes R graphics caches without writing images or preparing Python tables.
- `COMPASS_FIGURE_SCOPE=federated` renders only the no-MSK PSA/testosterone forests at days 0/90/180, excluding deltas and showing nominal p < 0.05 and supplied FDR q < 0.05 significance.

After updating analysis inputs, rerun this notebook before the next ordinary knit. For forced table regeneration set `FORCE = True` above; the R overwrite switches only affect R graphics and images.
